# Wrocław Tram Derailments — Scraper

**Co robi ten notebook:**  
Pobiera listę artykułów ze strony wroclaw.pl, filtruje te o wykolejeniach tramwajów i zapisuje surowe dane do CSV.

**Pipeline:**  
`sitemap.xml → lista URL-i → każdy artykuł → data + tytuł + treść → incidents_raw.csv`

## 1. Importy

Importujemy biblioteki których będziemy używać.  
- `requests` — wysyła zapytania HTTP (pobiera stronę jak przeglądarka)  
- `BeautifulSoup` — parsuje HTML i pozwala wyciągać dane z tagów  
- `pandas` — tworzy tabele (DataFrame) i zapisuje do CSV  
- `time.sleep` — pauza między zapytaniami (żeby nie przeciążać serwera — dobra praktyka)  
- `re` — wyrażenia regularne, do wyciągania linii tramwajowej z tekstu

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from time import sleep
import re

print("Biblioteki załadowane.")

## 2. Pobieranie listy URL-i z sitemapa

Sitemap to plik XML który każda strona publikuje dla wyszukiwarek — lista wszystkich podstron.  
Zamiast klikać "następna strona" 72 razy, pobieramy od razu wszystkie 718 URL-i z jednego pliku.

`headers` — podajemy się za przeglądarkę Firefox. Niektóre serwery blokują zapytania bez tego nagłówka.

In [ ]:
SITEMAP_URL = "https://www.wroclaw.pl/komunikacja/sitemap.xml"

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"
}

response = requests.get(SITEMAP_URL, headers=headers)
print(f"Status: {response.status_code}")  # 200 = sukces, 404 = nie znaleziono
print(f"Rozmiar odpowiedzi: {len(response.text)} znaków")

## 3. Parsowanie XML — wyciągamy URL-e

Sitemap jest w formacie XML. BeautifulSoup potrafi go czytać tak samo jak HTML.  
Szukamy tagów `<loc>` — w każdym jest jeden URL artykułu.

In [ ]:
soup = BeautifulSoup(response.text, "xml")
all_urls = [tag.text for tag in soup.find_all("loc")]

print(f"Łączna liczba URL-i w sitemapie: {len(all_urls)}")
print("\nPierwsze 5 URL-i:")
for url in all_urls[:5]:
    print(" ", url)

## 4. Filtrowanie — tylko artykuły o tramwajach

Nie wszystkie 718 artykułów dotyczy wykolejenia — są też artykuły o rowerach, parkingach, itp.  
Pierwsza selekcja po URL-u: zostawiamy tylko te które mają w adresie słowa powiązane z tramwajem.

To nie wystarczy — w kroku 5 będziemy sprawdzać też treść artykułu.

In [ ]:
TRAM_KEYWORDS = ["wykolejenie", "tramwaj", "tramwaje", "rozjechanie"]

tram_urls = [
    url for url in all_urls
    if any(keyword in url.lower() for keyword in TRAM_KEYWORDS)
]

print(f"Artykuły z tramwajem w URL: {len(tram_urls)} z {len(all_urls)}")
print("\nPrzykłady:")
for url in tram_urls[:10]:
    print(" ", url)

## 5. Scraping artykułów — pętla przez URL-e

Dla każdego URL-a:  
1. Pobieramy stronę (`requests.get`)  
2. Parsujemy HTML (`BeautifulSoup`)  
3. Wyciągamy: **datę**, **tytuł**, **pełny tekst**  
4. Sprawdzamy czy treść zawiera słowo "wykolejenie" (drugi filtr)  
5. Zapisujemy do listy wyników  

`sleep(1)` — czekamy 1 sekundę między zapytaniami. To ważne: bez tego serwer może zablokować nasz IP jako robota. Grzeczny scraper nie przeciąża serwera.

In [ ]:
results = []

for i, url in enumerate(tram_urls):
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")

        # Tytuł artykułu
        title_tag = soup.find("h1")
        title = title_tag.text.strip() if title_tag else ""

        # Data publikacji — wroclaw.pl używa tagu <time> z atrybutem datetime
        time_tag = soup.find("time")
        date = time_tag.get("datetime", "") if time_tag else ""

        # Pełny tekst artykułu (do późniejszego wyciągania linii i ulic)
        article_tag = soup.find("article")
        text = article_tag.get_text(" ", strip=True) if article_tag else ""

        # Drugi filtr: czy treść faktycznie mówi o wykolejeniu?
        if "wykolejenie" not in text.lower() and "wykolejenie" not in title.lower():
            continue

        results.append({
            "url": url,
            "title": title,
            "date": date,
            "text": text
        })

        print(f"[{i+1}/{len(tram_urls)}] ✓ {title[:60]}")

    except Exception as e:
        print(f"[{i+1}/{len(tram_urls)}] ✗ Błąd: {url} → {e}")

    sleep(1)

print(f"\nZnaleziono artykułów o wykolejeniach: {len(results)}")

## 6. Zapis do CSV

`pandas.DataFrame` to tabela — tu zamieniamy listę słowników na tabelę.  
Zapisujemy do pliku CSV w folderze `data/` — to nasze surowe dane, nieprzetworzone.

Dobra praktyka DE: zawsze zachowuj surowe dane osobno (stąd nazwa `_raw`). Nigdy ich nie nadpisujesz — jak coś pójdzie nie tak w czyszczeniu, możesz wrócić do źródła.

In [ ]:
df = pd.DataFrame(results)

output_path = "../data/incidents_raw.csv"
df.to_csv(output_path, index=False, encoding="utf-8")

print(f"Zapisano {len(df)} wierszy do {output_path}")
print("\nPodgląd danych:")
df[["date", "title", "url"]].head(10)